# Laboratory Work #6
## Group: IT2-2404SE
## Full Name: Daniil Kalts

### Exercises – auto-mpg
http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/

1) Load auto-mpg.data into a DataFrame autodf.

2) Give description of the generated DataFrame autodf. (Description-> Main Statistics) .describe() OR aggregate()

3) Display the first 10 rows of the DataFrame autodf. head(10)

4) Find the attributes which have missing values. Handle the missing values using following two ways:

   i. Replace the missing values by a value before that. method = 'pad'

   ii. Remove the rows having missing values from the original dataset. dropna()

5) Print the details of the car which gave the maximum mileage.

6) Find the average displacement of the car given the number of cylinders. mean

7) What is the average number of cylinders in a car? [cylinder] -> mean

8) Determine the no. of cars with weight greater than the average weight. count

In [1]:
import pandas as pd

base_path = "Datasets/"

# 1)
columns = [
    "mpg", "cylinders", "displacement", "horsepower", "weight",
    "acceleration", "model_year", "origin", "car_name"
]

autodf = pd.read_csv(
    base_path + "auto-mpg.data",
    sep=r"\s+",
    names=columns,
    na_values="?"
)

print("1) auto-mpg loaded:", autodf.shape)
print()

# 2)
print("2) Main statistics:")
print(autodf.describe())
print()

# 3)
print("3) First 10 rows:")
print(autodf.head(10))
print()

# 4)
missing = autodf.isnull().sum()
missing_rows = autodf[autodf.isnull().any(axis=1)].index

print("4) Attributes with missing values:")
print(missing[missing > 0])
print()

# i.
autodf_pad = autodf.ffill()

print("4.i) Missing rows after 'pad' (forward fill):")
print(autodf_pad.loc[missing_rows, ["car_name", "horsepower"]])
print("Missing values left:", autodf_pad.isnull().sum().sum())
print()

# ii.
autodf = autodf.dropna()

print("4.ii) Shape after dropna():", autodf.shape)
print("Missing values left:", autodf.isnull().sum().sum())
print()

# 5)
print("5) Car with the maximum mileage:")
print(autodf.loc[autodf["mpg"].idxmax()])
print()

# 6)
print("6) Average displacement by number of cylinders:")
print(autodf.groupby("cylinders")["displacement"].mean())
print()

# 7)
print("7) Average number of cylinders:")
print(autodf["cylinders"].mean())
print()

# 8)
average_weight = autodf["weight"].mean()
heavier_cars = (autodf["weight"] > average_weight).sum()

print("8) Average weight:", average_weight)
print("Cars heavier than average:", heavier_cars)

1) auto-mpg loaded: (398, 9)

2) Main statistics:
              mpg   cylinders  displacement  horsepower       weight  \
count  398.000000  398.000000    398.000000  392.000000   398.000000   
mean    23.514573    5.454774    193.425879  104.469388  2970.424623   
std      7.815984    1.701004    104.269838   38.491160   846.841774   
min      9.000000    3.000000     68.000000   46.000000  1613.000000   
25%     17.500000    4.000000    104.250000   75.000000  2223.750000   
50%     23.000000    4.000000    148.500000   93.500000  2803.500000   
75%     29.000000    8.000000    262.000000  126.000000  3608.000000   
max     46.600000    8.000000    455.000000  230.000000  5140.000000   

       acceleration  model_year      origin  
count    398.000000  398.000000  398.000000  
mean      15.568090   76.010050    1.572864  
std        2.757689    3.697627    0.802055  
min        8.000000   70.000000    1.000000  
25%       13.825000   73.000000    1.000000  
50%       15.500000   76.

### Part 1 – Loading Data
You are given three files:

sales.csv (comma-separated, 1 million rows)

customers.json (nested JSON with customer info)

transactions.xlsx (Excel file with 3 sheets)

Tasks:
- a. Load all three files into pandas DataFrames.
- b. For the JSON file, normalize it so nested fields (like address.city) become separate columns.
- c. Merge the three DataFrames into a single one on customer_id.
- d. Report the total number of transactions per country.

In [2]:
import json
import pandas as pd

base_path = "Datasets/"

# a)
sales_df = pd.read_csv(base_path + "sales.csv")
sales_df["amount"] = pd.to_numeric(sales_df["amount"], errors="coerce")

customers_df = pd.read_json(base_path + "customers.json")

sheets = pd.read_excel(base_path + "transactions.xlsx", sheet_name=None)
transactions_df = pd.concat(
    [sheet.assign(month=sheet_name) for sheet_name, sheet in sheets.items()],
    ignore_index=True
)

print("a) Files loaded:")
print("Sales:", sales_df.shape)
print("Customers:", customers_df.shape)
for sheet_name, sheet in sheets.items():
    print(f"Transactions sheet '{sheet_name}':", sheet.shape)
print("Transactions (all sheets):", transactions_df.shape)
print()


# b)
with open(base_path + "customers.json") as file:
    customers_df = pd.json_normalize(json.load(file), sep=".")

print("b) Customer data after normalization:")
print(customers_df.head())
print()


# c)
all_transactions_df = pd.concat(
    [sales_df.assign(source="sales"), transactions_df.assign(source="transactions")],
    ignore_index=True
)

combined_df = pd.merge(
    all_transactions_df,
    customers_df,
    on="customer_id",
    how="inner"
)

print("c) Combined data:", combined_df.shape)
print(combined_df.head())
print()


# d)
transactions_per_country = (
    combined_df.groupby("address.country")
    .size()
    .reset_index(name="transaction_count")
    .sort_values("transaction_count", ascending=False, ignore_index=True)
)

print("d) Transactions per country:")
print(transactions_per_country)
print("Total:", transactions_per_country["transaction_count"].sum())

a) Files loaded:
Sales: (5002, 5)
Customers: (500, 4)
Transactions sheet 'Jan': (1000, 4)
Transactions sheet 'Feb': (1000, 4)
Transactions sheet 'Mar': (1000, 4)
Transactions (all sheets): (3000, 5)

b) Customer data after normalization:
  customer_id        name                  email address.city address.country
0       C0001  Customer_1  customer1@example.com       Sydney           Japan
1       C0002  Customer_2  customer2@example.com       Berlin       Australia
2       C0003  Customer_3  customer3@example.com       Sydney             USA
3       C0004  Customer_4  customer4@example.com     New York           Japan
4       C0005  Customer_5  customer5@example.com     New York          France

c) Combined data: (8000, 11)
  transaction_id customer_id product_id        date  amount source month  \
0         T00001       C0103       P107  2024-05-11  516.90  sales   NaN   
1         T00002       C0436       P186  2024-05-19  553.35  sales   NaN   
2         T00003       C0349       P

### Part 2 – Storage Decisions
You now have a combined DataFrame.
Tasks:
- a. Save the dataset in CSV and Pickle formats.
- b. Measure and compare the file sizes and loading times of each format.
- c. Which format would you recommend for:

    Archiving the dataset for future analysis
    
    Sharing with a colleague who does not use Python
    
    Fastest reloading inside Python

In [3]:
import pandas as pd
import time
import os

# a)
combined_df.to_csv("combined.csv", index=False)
combined_df.to_pickle("combined.pkl")

print("a) Dataset saved as CSV and Pickle.")
print()

# b)
start = time.perf_counter()
csv_file = pd.read_csv("combined.csv")
csv_time = time.perf_counter() - start

start = time.perf_counter()
pickle_file = pd.read_pickle("combined.pkl")
pickle_time = time.perf_counter() - start

csv_size = os.path.getsize("combined.csv")
pickle_size = os.path.getsize("combined.pkl")

print("b) File sizes and loading times:")
print("CSV file size:", csv_size, "bytes")
print("CSV loading time:", csv_time, "seconds")
print()

print("Pickle file size:", pickle_size, "bytes")
print("Pickle loading time:", pickle_time, "seconds")
print()

print("Smaller file:", "CSV" if csv_size < pickle_size else "Pickle")
print("Faster loading:", "CSV" if csv_time < pickle_time else "Pickle",
      f"({max(csv_time, pickle_time) / min(csv_time, pickle_time):.1f}x faster)")
print()

# c)
print("c) Storage recommendations:")
print()

# Q: Archiving the dataset for future analysis
# A: CSV, because it is plain text and can be opened by any tool or future version.
print("Archiving the dataset for future analysis:")
print("CSV, because it is plain text and can be opened by any tool or future version.")
print()

# Q: Sharing with a colleague who does not use Python
# A: CSV, because it opens in Excel and almost any other program.
print("Sharing with a colleague who does not use Python:")
print("CSV, because it opens in Excel and almost any other program.")
print()

# Q: Fastest reloading inside Python
# A: Pickle, it loads much faster because nothing has to be parsed.
print("Fastest reloading inside Python:")
print("Pickle, it loads much faster because nothing has to be parsed.")

a) Dataset saved as CSV and Pickle.

b) File sizes and loading times:
CSV file size: 744772 bytes
CSV loading time: 0.013870172999986607 seconds

Pickle file size: 1317318 bytes
Pickle loading time: 0.0009515869999177085 seconds

Smaller file: CSV
Faster loading: Pickle (14.6x faster)

c) Storage recommendations:

Archiving the dataset for future analysis:
CSV, because it is plain text and can be opened by any tool or future version.

Sharing with a colleague who does not use Python:
CSV, because it opens in Excel and almost any other program.

Fastest reloading inside Python:
Pickle, it loads much faster because nothing has to be parsed.


### Part 3 – Database Integration
Create a SQLite database called company.db.
Tasks:
- a. Write the merged DataFrame into a SQL table transactions.
- b. Run an SQL query (using pd.read_sql) to find the top 5 customers with the highest total purchase value.
- c. Export the SQL query result into both Excel and JSON formats.

In [4]:
import sqlite3
import pandas as pd

# a)
connection = sqlite3.connect("company.db")

combined_df.to_sql(
    "transactions",
    connection,
    if_exists="replace",
    index=False
)

print("a) Data saved to company.db")
print()

# b)
query = """
SELECT
    customer_id,
    name,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_purchase
FROM transactions
GROUP BY customer_id, name
ORDER BY total_purchase DESC
LIMIT 5
"""

top_customers = pd.read_sql(query, connection)

print("b) Top 5 customers by total purchase value:")
print(top_customers)
print()

# c)
top_customers.to_excel("top_customers.xlsx", index=False)
top_customers.to_json("top_customers.json", orient="records", indent=2)

print("c) Results saved to top_customers.xlsx and top_customers.json")

connection.close()

a) Data saved to company.db

b) Top 5 customers by total purchase value:
  customer_id          name  transaction_count  total_purchase
0       C0461  Customer_461                 24        20445.29
1       C0106  Customer_106                 23        20300.62
2       C0435  Customer_435                 30        20004.97
3       C0395  Customer_395                 22        19847.61
4       C0271  Customer_271                 22        19680.18

c) Results saved to top_customers.xlsx and top_customers.json


### Part 4 – Advanced Challenge
Sometimes we need to handle corrupt/malformed data. Suppose sales.csv has some missing values and extra delimiters.
Tasks:
- a. Load sales.csv with parameters that handle bad lines gracefully (error_bad_lines, na_values, etc.).
- b. Replace missing values in the amount column with the column median.
- c. Save the cleaned file in Feather format.

In [5]:
import pandas as pd

# a)
sales_df = pd.read_csv(
    base_path + "sales.csv",
    on_bad_lines="skip",
    na_values=["", "NA", "NULL", "?", "not_a_number"]
)

print("a) Rows loaded:", len(sales_df))
print("Missing values per column:")
print(sales_df.isna().sum())
print()

sales_df["date"] = pd.to_datetime(sales_df["date"], format="%Y-%m-%d", errors="coerce")
malformed = sales_df["date"].isna()

print("Malformed rows removed:")
print(sales_df[malformed])
print()

sales_df = sales_df[~malformed].reset_index(drop=True)
print("Rows after cleaning:", len(sales_df))
print()

# b)
sales_df["amount"] = pd.to_numeric(sales_df["amount"], errors="coerce")
median_amount = sales_df["amount"].median()

print("b) Missing amounts before:", sales_df["amount"].isna().sum())
sales_df["amount"] = sales_df["amount"].fillna(median_amount)

print("Median amount:", median_amount)
print("Missing amounts after:", sales_df["amount"].isna().sum())
print()

# c)
sales_df.to_feather(base_path + "cleaned_sales.feather")

print("c) Saved cleaned_sales.feather")
print(pd.read_feather(base_path + "cleaned_sales.feather").dtypes)

a) Rows loaded: 5002
Missing values per column:
transaction_id     0
customer_id        0
product_id         0
date               1
amount            52
dtype: int64

Malformed rows removed:
     transaction_id customer_id product_id date  amount
5000        BADLINE       EXTRA       DATA  NaT     NaN
5001         T99999       C9999       P999  NaT     NaN

Rows after cleaning: 5000

b) Missing amounts before: 50
Median amount: 499.14
Missing amounts after: 0

c) Saved cleaned_sales.feather
transaction_id               str
customer_id                  str
product_id                   str
date              datetime64[us]
amount                   float64
dtype: object
